# Build the master GTEx MutPred2 score file

Assembles `MutPred2_score_gtex.tsv` — one MutPred2 score per (variant, transcript) pair for the
variants in `extractedVariants.xlsx`, drawn from two sources and nothing else:

| source | file | what it contributes |
|---|---|---|
| SJ's original run | `MutPred2_for_filtered_variants_SJ.tsv` | the bulk of the scores |
| Tim's rescoring run | `MutPred2_scores_missing_and_control.tsv` | the variants SJ was missing |

Both are scores we generated ourselves. **Tulika's files are not read here at all** — they are a
reference for what is missing, never a source of scores.

**`MutPred2_scores_ASD_brainspan.tsv` is deliberately excluded.** It is built on BrainSpan's
transcript set, whose isoform versions differ from GTEx's. ASD variants appear in both the GTEx
and the BrainSpan analyses, but the two must stay separate, so BrainSpan-based scores must never
enter a GTEx file. Section 4 verifies this rather than assuming it.

Output schema matches the existing `MutPred2_scores.tsv` exactly: `Variant, ENST, aa_sub, MutPred2`.

## 1. Setup

In [1]:
from pathlib import Path

import pandas as pd

from ExTSP.config import DATA_DIR, ExtractedVariants_file, VariantAnnotations_file, GTeX_file

DATA             = Path(str(DATA_DIR))
AI_GENERATED_DIR = DATA / "AI_generated"

SJ_FILE    = DATA / "MutPred2_scores" / "MutPred2_for_filtered_variants_SJ.tsv"
TIM_FILE   = DATA / "MutPred2_scores" / "MutPred2_scores_missing_and_control.tsv"
ASD_BRAINSPAN = DATA / "MutPred2_scores" / "MutPred2_scores_ASD_brainspan.tsv"   # excluded, checked in section 4

MASTER_OUT = AI_GENERATED_DIR / "MutPred2_score_gtex.tsv"

SHEETS = ["CM", "PCD", "PH", "PKD", "ASD", "nonTarget"]

# Keep only variants present in the workbook. Flip to False to keep every scored variant,
# including ones outside the current extraction.
RESTRICT_TO_WORKBOOK = True

for f in (SJ_FILE, TIM_FILE, ExtractedVariants_file, VariantAnnotations_file, GTeX_file):
    print(f"{'OK ' if Path(f).exists() else 'MISSING'}  {f}")
print(f"\noutput -> {MASTER_OUT}")

OK   /workspaces/ExTSP-default-claude/src/ExTSP/data/MutPred2_scores/MutPred2_for_filtered_variants_SJ.tsv
OK   /workspaces/ExTSP-default-claude/src/ExTSP/data/MutPred2_scores/MutPred2_scores_missing_and_control.tsv
OK   /workspaces/ExTSP-default-claude/src/ExTSP/data/filtered/extractedVariants.xlsx
OK   /workspaces/ExTSP-default-claude/src/ExTSP/data/filtered/variant_annotations.tsv
OK   /workspaces/ExTSP-default-claude/src/ExTSP/data/raw/GTeX_V8_short_read_transcripts_meanTPM_expression_across_tissue_samples_Gencodev26_without_tpm_filter.tsv

output -> /workspaces/ExTSP-default-claude/src/ExTSP/data/AI_generated/MutPred2_score_gtex.tsv


## 2. The variant universe

Every variant across the five disease sheets and the nonTarget sheet of `extractedVariants.xlsx`.
This is what the master file is meant to cover.

In [2]:
xl = pd.ExcelFile(ExtractedVariants_file)
sheets = {name: xl.parse(name) for name in SHEETS}

workbook = set()
for name in SHEETS:
    v = set(sheets[name]["Variant"].dropna().unique())
    workbook |= v
    print(f"  {name:<11} {len(v):>7} variants")

print(f"\nworkbook union : {len(workbook)} variants")

  CM            27540 variants
  PCD            8828 variants
  PH              414 variants
  PKD            2263 variants
  ASD            1425 variants
  nonTarget     19718 variants

workbook union : 60183 variants


## 3. The two score sources

**SJ's file** is keyed by `ID` (an ENST transcript) + `Substitution` and carries no genomic
variant column, so it has to be mapped through `variant_annotations.tsv` on (ENST, aa_sub).

One subtlety in that mapping: a handful of (transcript, substitution) keys correspond to more
than one genomic variant, because different nucleotide changes can produce the same amino-acid
substitution. The score is a property of the protein change, so every such variant legitimately
receives the same score — the mapping is allowed to fan out, and the count is reported below.

**Tim's file** already carries the genomic variant on each row, so it needs no mapping.

In [3]:
ann = (pd.read_csv(VariantAnnotations_file, sep="\t", low_memory=False)
         [["ENST", "aa_sub", "Variant"]].dropna().drop_duplicates())

fanout = ann.groupby(["ENST", "aa_sub"])["Variant"].nunique()
print(f"variant_annotations: {len(ann)} (ENST, aa_sub, Variant) triples")
print(f"  (ENST, aa_sub) keys mapping to >1 genomic variant : {(fanout > 1).sum()}")
print()

# --- source A: SJ, mapped onto genomic variants ---------------------------
sj = (pd.read_csv(SJ_FILE, sep="\t")
        .rename(columns={"ID": "ENST", "Substitution": "aa_sub"}))
sj = sj[sj["MutPred2"].notna()]
src_sj = ann.merge(sj, on=["ENST", "aa_sub"], how="inner")[["Variant", "ENST", "aa_sub", "MutPred2"]]

print(f"SJ file        : {len(sj)} scored (transcript, substitution) entries")
print(f"  mapped       : {len(src_sj)} rows, {src_sj['Variant'].nunique()} variants")
print(f"  UNMAPPED     : {len(sj) - len(sj.merge(ann, on=['ENST','aa_sub'], how='inner').drop_duplicates(['ENST','aa_sub']))}"
      f"  (no entry in variant_annotations)")

# --- source B: Tim, already variant-keyed ---------------------------------
tim = (pd.read_csv(TIM_FILE, sep="\t")
         .rename(columns={"ID": "Variant", "transcript": "ENST",
                          "Substitution": "aa_sub", "MutPred2 score": "MutPred2"}))
src_tim = tim[tim["MutPred2"].notna()][["Variant", "ENST", "aa_sub", "MutPred2"]]

print(f"\nTim file       : {len(src_tim)} scored rows, {src_tim['Variant'].nunique()} variants")

variant_annotations: 154860 (ENST, aa_sub, Variant) triples
  (ENST, aa_sub) keys mapping to >1 genomic variant : 668

SJ file        : 154114 scored (transcript, substitution) entries
  mapped       : 154801 rows, 49098 variants


  UNMAPPED     : 0  (no entry in variant_annotations)

Tim file       : 51099 scored rows, 12829 variants


## 4. Transcript-version guard

The reason BrainSpan scores are kept out of this file. Every transcript entering the master must
carry a transcript version that exists in the GTEx expression matrix (Gencode v26) — otherwise a
score would be attached to an isoform the GTEx side of the pipeline cannot match.

The same test is run on the ASD BrainSpan file to show concretely why it is excluded.

In [4]:
GTEX = set(pd.read_csv(GTeX_file, sep="\t", usecols=["transcript_id"])["transcript_id"])
GTEX_IDS = {t.split(".")[0] for t in GTEX}
print(f"GTEx (Gencode v26) transcripts: {len(GTEX)}\n")


def version_report(label, transcripts):
    t = set(transcripts)
    ids = {x.split(".")[0] for x in t}
    exact, by_id = len(t & GTEX), len(ids & GTEX_IDS)
    print(f"{label:<24} transcripts={len(t):>5}  exact version in GTEx={exact:>5}  "
          f"same ID, WRONG version={by_id - exact:>5}  absent={len(ids - GTEX_IDS):>5}")
    return exact == len(t)


ok_sj  = version_report("SJ (mapped)",   src_sj["ENST"])
ok_tim = version_report("Tim",           src_tim["ENST"])

asd_bs = pd.read_csv(ASD_BRAINSPAN, sep="\t")
version_report("ASD BrainSpan [EXCLUDED]", asd_bs["transcript"])

assert ok_sj and ok_tim, "a source carries transcript versions GTEx does not have"
print("\nBoth included sources are fully GTEx-versioned.")
print("The BrainSpan file is not - most of its transcripts are the same IDs at a different")
print("version, which is exactly why its scores must not be mixed into a GTEx file.")

GTEx (Gencode v26) transcripts: 199324

SJ (mapped)              transcripts=19900  exact version in GTEx=19900  same ID, WRONG version=    0  absent=    0
Tim                      transcripts= 1890  exact version in GTEx= 1890  same ID, WRONG version=    0  absent=    0
ASD BrainSpan [EXCLUDED] transcripts=  547  exact version in GTEx=   67  same ID, WRONG version=  470  absent=   10

Both included sources are fully GTEx-versioned.
The BrainSpan file is not - most of its transcripts are the same IDs at a different
version, which is exactly why its scores must not be mixed into a GTEx file.


## 5. Combine

Restricted to the workbook, then concatenated. Where SJ and Tim both scored the same
(variant, transcript, substitution) — the 1,000 positive controls — the two must agree; a
conflict is an error, not something to silently resolve, so it is asserted rather than averaged.

In [5]:
if RESTRICT_TO_WORKBOOK:
    dropped_sj  = src_sj[~src_sj["Variant"].isin(workbook)]
    dropped_tim = src_tim[~src_tim["Variant"].isin(workbook)]
    a = src_sj[src_sj["Variant"].isin(workbook)]
    b = src_tim[src_tim["Variant"].isin(workbook)]
    print(f"restricted to workbook variants:")
    print(f"  SJ  kept {len(a):>6} rows, dropped {len(dropped_sj):>6} "
          f"({dropped_sj['Variant'].nunique()} variants outside the workbook)")
    print(f"  Tim kept {len(b):>6} rows, dropped {len(dropped_tim):>6} "
          f"({dropped_tim['Variant'].nunique()} variants outside the workbook)")
else:
    a, b = src_sj, src_tim
    print("keeping every scored variant, workbook or not")

combined = pd.concat([a, b], ignore_index=True)

KEY = ["Variant", "ENST", "aa_sub"]
overlap = combined.duplicated(KEY).sum()
conflicts = combined.groupby(KEY)["MutPred2"].nunique()
n_conflict = int((conflicts > 1).sum())

print(f"\ncombined rows                       : {len(combined)}")
print(f"  keys scored by both sources       : {overlap}")
print(f"  keys where the two DISAGREE       : {n_conflict}")

assert n_conflict == 0, "SJ and Tim disagree on a shared (variant, transcript, substitution)"

restricted to workbook variants:
  SJ  kept 118522 rows, dropped  36279 (10652 variants outside the workbook)
  Tim kept  51099 rows, dropped      0 (0 variants outside the workbook)



combined rows                       : 169621
  keys scored by both sources       : 5341
  keys where the two DISAGREE       : 0


## 6. The master table

In [6]:
master = (combined.drop_duplicates(KEY, keep="first")
                  .sort_values(KEY)
                  .reset_index(drop=True))[["Variant", "ENST", "aa_sub", "MutPred2"]]

assert not master.duplicated(KEY).any()
assert master["Variant"].notna().all(), "a row has no genomic variant"
assert master["MutPred2"].notna().all(), "a row has no score"
assert master["MutPred2"].between(0, 1).all(), "a MutPred2 score falls outside [0, 1]"
assert master["ENST"].isin(GTEX).all(), "a transcript is not in the GTEx transcript set"

print(f"master rows        : {len(master)}")
print(f"  variants         : {master['Variant'].nunique()}")
print(f"  transcripts      : {master['ENST'].nunique()}")
print(f"  scores per variant (median) : {master.groupby('Variant').size().median():.0f}")
print()
master.head(10)

master rows        : 164280
  variants         : 50323
  transcripts      : 19864
  scores per variant (median) : 2



,Variant,ENST,aa_sub,MutPred2
0,10_100209695_C/T,ENST00000370397.7,R343H,0.080
1,10_100293374_C/A,ENST00000318222.3,M555I,0.822
2,10_100297026_C/T,ENST00000318222.3,R380H,0.053
3,10_100529393_C/T,ENST00000299166.8,D67N,0.531
4,10_100529393_C/T,ENST00000370320.4,D67N,0.170
5,10_100529393_C/T,ENST00000370322.5,D36N,0.161
6,10_100529393_C/T,ENST00000527595.5,D67N,0.151
7,10_100529393_C/T,ENST00000528174.5,D67N,0.114
8,10_100529393_C/T,ENST00000528425.5,D67N,0.114
9,10_100529393_C/T,ENST00000529568.5,D67N,0.215


### 6a. Coverage per sheet

In [7]:
covered = set(master["Variant"])

rows = []
for name in SHEETS:
    v = set(sheets[name]["Variant"].dropna().unique())
    rows.append({"sheet": name, "variants": len(v),
                 "scored": len(v & covered), "gap": len(v - covered)})
cov = pd.DataFrame(rows)
cov["pct"] = (100 * cov["scored"] / cov["variants"]).round(1)
cov.loc[len(cov)] = {"sheet": "ALL (union)", "variants": len(workbook),
                     "scored": len(workbook & covered), "gap": len(workbook - covered),
                     "pct": round(100 * len(workbook & covered) / len(workbook), 1)}
print(cov.to_string(index=False))

      sheet  variants  scored  gap  pct
         CM     27540   25084 2456 91.1
        PCD      8828    7929  899 89.8
         PH       414     340   74 82.1
        PKD      2263    1719  544 76.0
        ASD      1425     506  919 35.5
  nonTarget     19718   14750 4968 74.8
ALL (union)     60183   50323 9860 83.6


## 7. Write the file

Kept in `data/AI_generated/` for now, as requested, rather than overwriting anything the
pipeline currently points at.

In [8]:
master.to_csv(MASTER_OUT, sep="\t", index=False)

print(f"wrote {len(master)} rows to\n  {MASTER_OUT}\n")

check = pd.read_csv(MASTER_OUT, sep="\t")
print("columns :", list(check.columns))
print("rows    :", len(check))
assert list(check.columns) == ["Variant", "ENST", "aa_sub", "MutPred2"]
assert len(check) == len(master)
print("\nschema matches the existing MutPred2_scores.tsv layout.")

wrote 164280 rows to
  /workspaces/ExTSP-default-claude/src/ExTSP/data/AI_generated/MutPred2_score_gtex.tsv

columns : ['Variant', 'ENST', 'aa_sub', 'MutPred2']
rows    : 164280

schema matches the existing MutPred2_scores.tsv layout.


## 8. Summary

In [9]:
print("MutPred2_score_gtex.tsv")
print(f"  {len(master)} variant-transcript scores")
print(f"  {master['Variant'].nunique()} of {len(workbook)} workbook variants "
      f"({master['Variant'].nunique()/len(workbook):.1%})")
print(f"  {master['ENST'].nunique()} transcripts, all GTEx Gencode v26 versions")
print()
print("  sources : SJ run + Tim's missing/control run")
print("  excluded: Tulika's files (reference only), ASD BrainSpan (wrong transcript versions)")

MutPred2_score_gtex.tsv
  164280 variant-transcript scores
  50323 of 60183 workbook variants (83.6%)
  19864 transcripts, all GTEx Gencode v26 versions

  sources : SJ run + Tim's missing/control run
  excluded: Tulika's files (reference only), ASD BrainSpan (wrong transcript versions)
